# SignBridge - Dataset Exploration & Preprocessing

## Step 1: Core Statistics

In [3]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('..') / 'data' / 'how2sign_realigned_train.csv'

required_cols = [
    'VIDEO_ID',
    'VIDEO_NAME',
    'SENTENCE_ID',
    'SENTENCE_NAME',
    'START_REALIGNED',
    'END_REALIGNED',
    'SENTENCE',
]

df = pd.read_csv(DATA_PATH, sep='\t')
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df['START_REALIGNED'] = pd.to_numeric(df['START_REALIGNED'], errors='coerce')
df['END_REALIGNED'] = pd.to_numeric(df['END_REALIGNED'], errors='coerce')

df['duration_sec'] = df['END_REALIGNED'] - df['START_REALIGNED']
df['word_count'] = (
    df['SENTENCE']
    .fillna('')
    .astype(str)
    .str.strip()
    .str.split()
    .str.len()
)
df['char_count'] = df['SENTENCE'].fillna('').astype(str).str.len()

valid_duration = df['duration_sec'].dropna()
valid_words = df['word_count'].dropna()

summary = {
    'num_examples': len(df),
    'unique_videos': df['VIDEO_ID'].nunique(),
    'unique_sentences': df['SENTENCE_ID'].nunique(),
    'duplicate_sentence_ids': int(df.duplicated(subset=['SENTENCE_ID']).sum()),
    'missing_sentences': int(df['SENTENCE'].isna().sum()),
    'non_positive_duration_rows': int((df['duration_sec'] <= 0).sum()),
    'total_duration_hours': float(valid_duration.sum() / 3600.0),
    'avg_duration_sec': float(valid_duration.mean()),
    'median_duration_sec': float(valid_duration.median()),
    'p90_duration_sec': float(valid_duration.quantile(0.90)),
    'avg_word_count': float(valid_words.mean()),
    'median_word_count': float(valid_words.median()),
    'p90_word_count': float(valid_words.quantile(0.90)),
}

print('How2Sign realigned train - core stats')
print('-' * 44)
for k, v in summary.items():
    if isinstance(v, float):
        print(f'{k:28s}: {v:.3f}')
    else:
        print(f'{k:28s}: {v}')

print('\nDuration statistics (sec):')
print(valid_duration.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

print('\nWord-count statistics:')
print(valid_words.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

print('\nTop 5 longest examples by duration:')
display(
    df.sort_values('duration_sec', ascending=False)[
        ['VIDEO_ID', 'SENTENCE_ID', 'START_REALIGNED', 'END_REALIGNED', 'duration_sec', 'word_count', 'SENTENCE']
    ].head(5)
)


How2Sign realigned train - core stats
--------------------------------------------
num_examples                : 31165
unique_videos               : 2192
unique_sentences            : 30932
duplicate_sentence_ids      : 233
missing_sentences           : 0
non_positive_duration_rows  : 0
total_duration_hours        : 59.331
avg_duration_sec            : 6.854
median_duration_sec         : 5.270
p90_duration_sec            : 13.610
avg_word_count              : 17.659
median_word_count           : 15.000
p90_word_count              : 33.000

Duration statistics (sec):
count    31165.000000
mean         6.853542
std          6.325943
min          0.010000
10%          1.670000
25%          2.990000
50%          5.270000
75%          8.800000
90%         13.610000
95%         17.630000
99%         28.513600
max        143.030000
Name: duration_sec, dtype: float64

Word-count statistics:
count    31165.000000
mean        17.659394
std         12.636150
min          1.000000
10%          5.0

,VIDEO_ID,SENTENCE_ID,START_REALIGNED,END_REALIGNED,duration_sec,word_count,SENTENCE
24247,FIbFH_Tdnb8,FIbFH_Tdnb8_2,14.77,157.80,143.03,19,"So I'm going to show it to you a few times, an..."
3127,05svBr3FJD4,05svBr3FJD4_1,0.75,124.30,123.55,195,This one is called X marks the spot and it's a...
9123,1LEWQlRiIDg,1LEWQlRiIDg_10,0.85,124.00,123.15,24,And women were considered to be better than me...
8857,1IHiA_6XXrI,1IHiA_6XXrI_12,1.97,124.59,122.62,11,They're kind of like big poofy baggy pants for...
605,-916rCqIrfY,-916rCqIrfY_20,107.00,228.56,121.56,17,"But, I think it's a great way to kick, bring i..."


## Step 2: Clip-to-Annotation Matching

In [9]:
from pathlib import Path
import pandas as pd

CLIPS_DIR = Path(r'D:\How2Sign\train\train_rgb_front_clips')
if 'df' not in globals():
    df = pd.read_csv(Path('..') / 'data' / 'how2sign_realigned_train.csv', sep='\t')

sentence_names = set(df['SENTENCE_NAME'].astype(str).tolist())
clip_files = [p for p in CLIPS_DIR.glob('*.mp4')]
clip_names = set(p.stem for p in clip_files)

matched = sentence_names & clip_names
missing_clips = sentence_names - clip_names
extra_files = clip_names - sentence_names

print('Clip-to-annotation matching')
print('-' * 32)
print(f'Total annotations (sentences): {len(sentence_names)}')
print(f'Total clip files: {len(clip_names)}')
print(f'Matched: {len(matched)}')
print(f'Missing clips: {len(missing_clips)}')
print(f'Unmatched clip files (no annotation): {len(extra_files)}')

if missing_clips:
    print('\nSample missing clips (up to 10):')
    for name in list(sorted(missing_clips))[:10]:
        print(f'  {name}')

if extra_files:
    print('\nSample extra clip files (up to 10):')
    for name in list(sorted(extra_files))[:10]:
        print(f'  {name}')


Clip-to-annotation matching
--------------------------------
Total annotations (sentences): 31165
Total clip files: 31047
Matched: 31047
Missing clips: 118
Unmatched clip files (no annotation): 0

Sample missing clips (up to 10):
  -b44ieHORm4_8-5-rgb_front
  006DXpJ9erw_8-5-rgb_front
  08JotxFsA4Y_11-5-rgb_front
  09KiqMdCSKc_22-8-rgb_front
  0E4mKYwAhOs_8-8-rgb_front
  0Esq_TAc-1Y_19-5-rgb_front
  0Esq_TAc-1Y_20-5-rgb_front
  0Esq_TAc-1Y_21-5-rgb_front
  0Esq_TAc-1Y_22-5-rgb_front
  0Esq_TAc-1Y_23-5-rgb_front


## Step 3: Drop Annotations with Missing Clips

In [10]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('..') / 'data' / 'how2sign_realigned_train.csv'
FILTERED_OUT = Path('..') / 'data' / 'how2sign_realigned_train_matched_clips.csv'
MISSING_OUT = Path('..') / 'data' / 'how2sign_missing_clips.csv'

if 'df' not in globals():
    df = pd.read_csv(DATA_PATH, sep='\t')

CLIPS_DIR = Path(r'D:\How2Sign\train\train_rgb_front_clips')
clip_names = set(p.stem for p in CLIPS_DIR.glob('*.mp4'))

missing_mask = ~df['SENTENCE_NAME'].astype(str).isin(clip_names)
missing_df = df[missing_mask].copy()
filtered_df = df[~missing_mask].copy()

missing_df[['SENTENCE_NAME']].drop_duplicates().to_csv(MISSING_OUT, index=False)
filtered_df.to_csv(FILTERED_OUT, sep='\t', index=False)

print('Dropped annotations with missing clips')
print('-' * 40)
print(f'Original rows: {len(df)}')
print(f'Missing clip rows dropped: {len(missing_df)}')
print(f'Filtered rows: {len(filtered_df)}')
print(f'Saved filtered CSV: {FILTERED_OUT.resolve()}')
print(f'Saved missing list: {MISSING_OUT.resolve()}')


Dropped annotations with missing clips
----------------------------------------
Original rows: 31165
Missing clip rows dropped: 118
Filtered rows: 31047
Saved filtered CSV: C:\My Projects\sign-language-bridge\data\how2sign_realigned_train_matched_clips.csv
Saved missing list: C:\My Projects\sign-language-bridge\data\how2sign_missing_clips.csv


## Step 3.1: Analysis - Impact of Removing Longest 10%

In [5]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path('..') / 'data' / 'how2sign_realigned_train_matched_clips.csv'

df = pd.read_csv(CSV_PATH, sep='\t')
df['duration_sec'] = df['END_REALIGNED'] - df['START_REALIGNED']
df['word_count'] = df['SENTENCE'].fillna('').astype(str).str.strip().str.split().str.len()

# Calculate p90 threshold
p90_threshold = df['duration_sec'].quantile(0.90)

# Split into keep vs remove
keep_mask = df['duration_sec'] <= p90_threshold
remove_mask = df['duration_sec'] > p90_threshold

keep_df = df[keep_mask]
remove_df = df[remove_mask]

# Calculate stats
total_examples = len(df)
total_hours = df['duration_sec'].sum() / 3600.0

keep_examples = len(keep_df)
keep_hours = keep_df['duration_sec'].sum() / 3600.0

remove_examples = len(remove_df)
remove_hours = remove_df['duration_sec'].sum() / 3600.0

print('Impact Analysis: Removing Clips > p90 (Longest 10%)')
print('=' * 56)
print(f'P90 threshold: {p90_threshold:.2f} seconds')
print()
print('Original dataset:')
print(f'  Examples: {total_examples:,}')
print(f'  Total hours: {total_hours:.2f}')
print()
print('After filtering (keeping <= p90):')
print(f'  Examples: {keep_examples:,} ({keep_examples/total_examples*100:.1f}%)')
print(f'  Total hours: {keep_hours:.2f} ({keep_hours/total_hours*100:.1f}%)')
print()
print('Removed (> p90):')
print(f'  Examples: {remove_examples:,} ({remove_examples/total_examples*100:.1f}%)')
print(f'  Total hours: {remove_hours:.2f} ({remove_hours/total_hours*100:.1f}%)')
print()
print('Duration stats after filtering:')
print(f'  Mean: {keep_df["duration_sec"].mean():.2f} sec')
print(f'  Median: {keep_df["duration_sec"].median():.2f} sec')
print(f'  Max: {keep_df["duration_sec"].max():.2f} sec')
print()
print('Sample removed clips (longest):')
display(remove_df.nlargest(10, 'duration_sec')[['SENTENCE_NAME', 'duration_sec', 'word_count', 'SENTENCE']])

Impact Analysis: Removing Clips > p90 (Longest 10%)
P90 threshold: 13.62 seconds

Original dataset:
  Examples: 31,047
  Total hours: 59.21

After filtering (keeping <= p90):
  Examples: 27,943 (90.0%)
  Total hours: 41.52 (70.1%)

Removed (> p90):
  Examples: 3,104 (10.0%)
  Total hours: 17.69 (29.9%)

Duration stats after filtering:
  Mean: 5.35 sec
  Median: 4.78 sec
  Max: 13.62 sec

Sample removed clips (longest):


,SENTENCE_NAME,duration_sec,word_count,SENTENCE
24141,FIbFH_Tdnb8_2-1-rgb_front,143.03,19,"So I'm going to show it to you a few times, an..."
3125,05svBr3FJD4_1-5-rgb_front,123.55,195,This one is called X marks the spot and it's a...
9092,1LEWQlRiIDg_10-5-rgb_front,123.15,24,And women were considered to be better than me...
8827,1IHiA_6XXrI_12-5-rgb_front,122.62,11,They're kind of like big poofy baggy pants for...
605,-916rCqIrfY_20-5-rgb_front,121.56,17,"But, I think it's a great way to kick, bring i..."
9158,1Lm_fpAjM9g_0-5-rgb_front,108.93,36,O'kay to accompany what we have been doing bef...
8756,1H-0xMsyUNA_10-5-rgb_front,108.16,38,"Get your oven to 350, put it in a nice preset ..."
9078,1KoWaCTX1jU_3-5-rgb_front,107.02,15,"Today, we're going to talk about something tha..."
14496,1zQYQrcYbrs_4-5-rgb_front,94.11,18,You want to make sure also that it is firm all...
8970,1J-XWwbc53M_11-5-rgb_front,86.24,19,And they could be used for all different types...


## Step 3.2: Filter Dataset - Remove Longest 10%

In [6]:
from pathlib import Path
import pandas as pd

CSV_IN  = Path('..') / 'data' / 'how2sign_realigned_train_matched_clips.csv'
CSV_OUT = Path('..') / 'data' / 'how2sign_realigned_train_filtered.csv'
REMOVED_OUT = Path('..') / 'data' / 'how2sign_removed_long_clips.csv'

EXECUTE_FILTER = True  # Set to True to actually filter

df = pd.read_csv(CSV_IN, sep='\t')
df['duration_sec'] = df['END_REALIGNED'] - df['START_REALIGNED']
df['word_count'] = df['SENTENCE'].fillna('').astype(str).str.strip().str.split().str.len()

# Calculate p90 threshold
p90_threshold = df['duration_sec'].quantile(0.90)

# Split
keep_df = df[df['duration_sec'] <= p90_threshold].copy()
remove_df = df[df['duration_sec'] > p90_threshold].copy()

if EXECUTE_FILTER:
    # Save filtered dataset
    keep_df.to_csv(CSV_OUT, sep='\t', index=False)
    
    # Save removed clips list (for reference, video files stay on disk)
    remove_df[['SENTENCE_NAME', 'duration_sec', 'word_count']].to_csv(REMOVED_OUT, index=False)
    
    print('Filter applied!')
    print(f'  Filtered CSV saved: {CSV_OUT.resolve()}')
    print(f'  Removed list saved: {REMOVED_OUT.resolve()}')
    print(f'\nKept {len(keep_df):,} examples (<= {p90_threshold:.2f} sec)')
    print(f'Removed {len(remove_df):,} examples (> {p90_threshold:.2f} sec)')
    print('\nNote: Video files on disk are NOT deleted, only CSV entries removed.')
else:
    print('EXECUTE_FILTER = False — no changes made.')
    print('Review the analysis above, then set EXECUTE_FILTER = True to proceed.')

Filter applied!
  Filtered CSV saved: C:\My Projects\sign-language-bridge\data\how2sign_realigned_train_filtered.csv
  Removed list saved: C:\My Projects\sign-language-bridge\data\how2sign_removed_long_clips.csv

Kept 27,943 examples (<= 13.62 sec)
Removed 3,104 examples (> 13.62 sec)

Note: Video files on disk are NOT deleted, only CSV entries removed.


## Step 4: MediaPipe Keypoint Extraction

In [ ]:
from pathlib import Path
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---- Config ----
CLIPS_DIR   = Path(r'D:\How2Sign\train\train_rgb_front_clips')
OUT_DIR     = Path('..') / 'data' / 'keypoints' / 'train'
CSV_PATH    = Path('..') / 'data' / 'how2sign_realigned_train_filtered.csv'
SAMPLE_SIZE = None  # None = full dataset, int = test on subset
FAILED_OUT  = Path('..') / 'data' / 'keypoint_extraction_failed.csv'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OVERWRITE   = False
SAVE_FORMAT = 'npz'
TARGET_FPS  = 20    # None = keep original fps
NUM_WORKERS = 4     # number of parallel threads

# ---- Landmark index sets ----
mp_fm = mp.solutions.face_mesh

def _idx(connection_set):
    return sorted({i for pair in connection_set for i in pair})

# Face subset controls
FACE_USE_LIPS      = True
FACE_EYE_MODE      = 'none'   # 'full' | 'iris' | 'none' - set to 'none' to exclude eyes
FACE_USE_EYEBROWS  = True
FACE_USE_NOSE      = False
NOSE_MODE          = 'none'   # 'full' | 'reduced' | 'none'
NOSE_MAX_POINTS    = 4

def _select_evenly(idx_list, k):
    if k is None or k >= len(idx_list): return idx_list
    if k <= 0: return []
    import numpy as _np
    sel = _np.linspace(0, len(idx_list) - 1, k)
    return [idx_list[int(round(i))] for i in sel]

REFINE_FACE_LANDMARKS = (FACE_EYE_MODE == 'iris')

face_parts = []
if FACE_USE_LIPS:
    face_parts += _idx(mp_fm.FACEMESH_LIPS)
if FACE_EYE_MODE == 'full':
    face_parts += _idx(mp_fm.FACEMESH_LEFT_EYE) + _idx(mp_fm.FACEMESH_RIGHT_EYE)
elif FACE_EYE_MODE == 'iris':
    face_parts += _idx(mp_fm.FACEMESH_IRISES)
if FACE_USE_EYEBROWS:
    face_parts += _idx(mp_fm.FACEMESH_LEFT_EYEBROW) + _idx(mp_fm.FACEMESH_RIGHT_EYEBROW)
nose_idx = []
if FACE_USE_NOSE and NOSE_MODE != 'none':
    nose_idx = _idx(mp_fm.FACEMESH_NOSE)
    if NOSE_MODE == 'reduced':
        nose_idx = _select_evenly(nose_idx, NOSE_MAX_POINTS)
    face_parts += nose_idx

FACE_IDX = sorted(set(face_parts))

# Pose: upper body only (shoulders=11-12, elbows=13-14, wrists=15-16,
# hand connection points=17-22, hip anchors=23-24)
# Removed index 0 (nose/face center) to avoid central dot on face
POSE_IDX = list(range(11, 25))  # 14 landmarks

nose_count = len(set(nose_idx)) if (FACE_USE_NOSE and NOSE_MODE != 'none') else 0
print('Face subset counts')
print(f'  lips: {len(set(_idx(mp_fm.FACEMESH_LIPS))) if FACE_USE_LIPS else 0}')
if FACE_EYE_MODE == 'full':
    eye_count = len(set(_idx(mp_fm.FACEMESH_LEFT_EYE) + _idx(mp_fm.FACEMESH_RIGHT_EYE)))
elif FACE_EYE_MODE == 'iris':
    eye_count = len(set(_idx(mp_fm.FACEMESH_IRISES)))
else:
    eye_count = 0
print(f'  eyes: {eye_count} (mode={FACE_EYE_MODE})')
print(f'  eyebrows: {len(set(_idx(mp_fm.FACEMESH_LEFT_EYEBROW) + _idx(mp_fm.FACEMESH_RIGHT_EYEBROW))) if FACE_USE_EYEBROWS else 0}')
print(f'  nose: {nose_count}')
print(f'Face landmarks selected : {len(FACE_IDX)}')
print(f'Pose landmarks selected : {len(POSE_IDX)}')
print(f'Hand landmarks (each)   : 21')
print(f'Total per frame         : {len(FACE_IDX) + len(POSE_IDX) + 21*2}')
print(f'Workers                 : {NUM_WORKERS}')

# ---- Worker function for parallel extraction ----
def extract_clip_keypoints(row, clips_dir, out_dir, target_fps, save_fmt, overwrite, 
                           face_idx, pose_idx, refine_face):
    """Extract keypoints for a single clip. Returns (sentence_name, success, error_msg)"""
    sentence_name = row.SENTENCE_NAME
    out_path = out_dir / f'{sentence_name}.{save_fmt}'
    
    if out_path.exists() and not overwrite:
        return (sentence_name, True, None)
    
    clip_path = clips_dir / f'{sentence_name}.mp4'
    if not clip_path.exists():
        return (sentence_name, False, 'clip_missing')
    
    try:
        cap = cv2.VideoCapture(str(clip_path))
        if not cap.isOpened():
            cap.release()
            return (sentence_name, False, 'cv2_open_failed')
        
        src_fps = cap.get(cv2.CAP_PROP_FPS)
        frame_interval = 1
        if target_fps and src_fps and src_fps > 0:
            frame_interval = max(int(round(src_fps / target_fps)), 1)
        
        frames_kp = []
        frames_mask = []
        frame_idx = 0
        H = W = None
        
        # Each worker gets its own MediaPipe instance
        mp_holistic = mp.solutions.holistic
        with mp_holistic.Holistic(
            static_image_mode=False,
            model_complexity=0,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
            refine_face_landmarks=refine_face,
        ) as holistic:
            
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                
                if frame_interval > 1 and (frame_idx % frame_interval) != 0:
                    frame_idx += 1
                    continue
                
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                res = holistic.process(rgb)
                
                H, W = frame.shape[:2]
                kp = []
                mask = []
                
                # Face (safe indexing)
                if res.face_landmarks:
                    lms = res.face_landmarks.landmark
                    lms_len = len(lms)
                    for i in face_idx:
                        if i < lms_len:
                            lm = lms[i]
                            kp.extend([lm.x, lm.y, lm.z])
                            mask.append(1)
                        else:
                            kp.extend([float('nan')] * 3)
                            mask.append(0)
                else:
                    kp.extend([float('nan')] * (len(face_idx) * 3))
                    mask.extend([0] * len(face_idx))
                
                # Pose
                if res.pose_landmarks:
                    lms = res.pose_landmarks.landmark
                    for i in pose_idx:
                        lm = lms[i]
                        kp.extend([lm.x, lm.y, lm.z])
                        mask.append(1)
                else:
                    kp.extend([float('nan')] * (len(pose_idx) * 3))
                    mask.extend([0] * len(pose_idx))
                
                # Hands
                for hand_lms in [res.left_hand_landmarks, res.right_hand_landmarks]:
                    if hand_lms:
                        for lm in hand_lms.landmark:
                            kp.extend([lm.x, lm.y, lm.z])
                        mask.extend([1] * 21)
                    else:
                        kp.extend([float('nan')] * (21 * 3))
                        mask.extend([0] * 21)
                
                frames_kp.append(kp)
                frames_mask.append(mask)
                frame_idx += 1
        
        cap.release()
        
        if not frames_kp:
            return (sentence_name, False, 'no_frames')
        
        # Save
        N = len(face_idx) + len(pose_idx) + 42
        arr = np.array(frames_kp, dtype=np.float32).reshape(len(frames_kp), N, 3)
        msk = np.array(frames_mask, dtype=np.uint8)
        np.savez(
            out_path,
            keypoints=arr,
            mask=msk,
            fps=np.array([float(src_fps)], dtype=np.float32) if (src_fps and src_fps > 0) else np.array([0.0], dtype=np.float32),
            image_size=np.array([H, W], dtype=np.int32) if (H and W) else np.array([0, 0], dtype=np.int32),
            face_idx=np.array(face_idx, dtype=np.int32),
            pose_idx=np.array(pose_idx, dtype=np.int32),
        )
        
        return (sentence_name, True, None)
    
    except Exception as e:
        return (sentence_name, False, str(e))

# ---- Main extraction ----
df = pd.read_csv(CSV_PATH, sep='\t')
if SAMPLE_SIZE is not None:
    df = df.head(int(SAMPLE_SIZE)).copy()

failed_list = []
success_count = 0

print(f'\nExtracting keypoints from {len(df)} clips...')

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {
        executor.submit(
            extract_clip_keypoints,
            row, CLIPS_DIR, OUT_DIR, TARGET_FPS, SAVE_FORMAT, OVERWRITE,
            FACE_IDX, POSE_IDX, REFINE_FACE_LANDMARKS
        ): idx
        for idx, row in enumerate(df.itertuples(index=False))
    }
    
    with tqdm(total=len(futures), desc='Extracting') as pbar:
        for future in as_completed(futures):
            sentence_name, success, error = future.result()
            if success:
                success_count += 1
            else:
                failed_list.append({'SENTENCE_NAME': sentence_name, 'error': error})
            pbar.update(1)

print(f'\nExtraction complete:')
print(f'  Successful: {success_count}')
print(f'  Failed: {len(failed_list)}')

if failed_list:
    failed_df = pd.DataFrame(failed_list)
    failed_df.to_csv(FAILED_OUT, index=False)
    print(f'  Failed list: {FAILED_OUT.resolve()}')
    print(f'\nSample failures:')
    display(failed_df.head(10))

print(f'\nDone. Keypoints saved to: {OUT_DIR.resolve()}')
print(f'Expected shape per file : (num_frames, {len(FACE_IDX) + len(POSE_IDX) + 42}, 3)')

Face subset counts
  lips: 40
  eyes: 0 (mode=none)
  eyebrows: 20
  nose: 0
Face landmarks selected : 60
Pose landmarks selected : 14
Hand landmarks (each)   : 21
Total per frame         : 116
Workers                 : 4

Extracting keypoints from 27943 clips...


Extracting:  67%|██████▋   | 18649/27943 [00:01<00:00, 13082.70it/s]c:\ProgramData\anaconda3\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Extracting:  69%|██████▊   | 19162/27943 [07:30<4:31:19,  1.85s/it] 

## Step 5: Overlay Keypoints on Sample Clips

In [ ]:
# from pathlib import Path
# import cv2
# import numpy as np
# import pandas as pd
# from tqdm import tqdm

# # ---- Config ----
# CLIPS_DIR    = Path(r'D:\How2Sign\train\train_rgb_front_clips')
# CSV_PATH     = Path('..') / 'data' / 'how2sign_realigned_train_filtered.csv'
# KEYPOINT_DIR = Path('..') / 'data' / 'keypoints' / 'train'
# KEYPOINT_EXT = 'npz'
# OUT_DIR      = Path('..') / 'data' / 'overlays'
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# SAMPLE_SIZE  = 5
# SAMPLE_NAMES = None   # set to a list of SENTENCE_NAME strings to override
# MAX_FRAMES   = None
# TARGET_FPS   = None
# OVERWRITE    = False

# # Choose samples
# if SAMPLE_NAMES is None:
#     df = pd.read_csv(CSV_PATH, sep='\t')
#     SAMPLE_NAMES = df['SENTENCE_NAME'].astype(str).head(int(SAMPLE_SIZE)).tolist()

# print(f'Overlaying {len(SAMPLE_NAMES)} clips')
# print(f'Keypoint dir: {KEYPOINT_DIR.resolve()}')
# print(f'Output dir: {OUT_DIR.resolve()}')

# POSE_CONNECTIONS = None
# HAND_CONNECTIONS = None
# try:
#     import mediapipe as mp
#     POSE_CONNECTIONS = mp.solutions.pose.POSE_CONNECTIONS
#     HAND_CONNECTIONS = mp.solutions.hands.HAND_CONNECTIONS
# except Exception:
#     pass


# def draw_points(frame, pts, mask, color, radius=2):
#     for i, p in enumerate(pts):
#         if mask is not None and mask[i] == 0:
#             continue
#         x, y = int(p[0] * frame.shape[1]), int(p[1] * frame.shape[0])
#         if 0 <= x < frame.shape[1] and 0 <= y < frame.shape[0]:
#             cv2.circle(frame, (x, y), radius, color, -1)


# def draw_connections(frame, pts, mask, connections, color, thickness=2):
#     for a, b in connections:
#         if mask is not None and (mask[a] == 0 or mask[b] == 0):
#             continue
#         xa, ya = int(pts[a][0] * frame.shape[1]), int(pts[a][1] * frame.shape[0])
#         xb, yb = int(pts[b][0] * frame.shape[1]), int(pts[b][1] * frame.shape[0])
#         if 0 <= xa < frame.shape[1] and 0 <= ya < frame.shape[0] and 0 <= xb < frame.shape[1] and 0 <= yb < frame.shape[0]:
#             cv2.line(frame, (xa, ya), (xb, yb), color, thickness)


# for name in tqdm(SAMPLE_NAMES, desc='Overlay'):
#     in_path  = CLIPS_DIR / f'{name}.mp4'
#     out_path = OUT_DIR / f'{name}_overlay.mp4'
#     kp_path  = KEYPOINT_DIR / f'{name}.{KEYPOINT_EXT}'

#     if not in_path.exists():
#         print(f'Skip missing clip: {name}')
#         continue
#     if not kp_path.exists():
#         print(f'Skip missing keypoints: {name}')
#         continue
#     if out_path.exists() and not OVERWRITE:
#         continue

#     data = np.load(kp_path, allow_pickle=True)
#     keypoints = data['keypoints']
#     mask      = data['mask'] if 'mask' in data.files else None
#     face_idx  = data['face_idx'] if 'face_idx' in data.files else None
#     pose_idx  = data['pose_idx'] if 'pose_idx' in data.files else None

#     T, N, _ = keypoints.shape
    
#     # Dynamically determine split sizes from saved metadata
#     face_len = int(len(face_idx)) if face_idx is not None else int(N - 14 - 42)
#     pose_len = int(len(pose_idx)) if pose_idx is not None else 14
#     hand_len = 21

#     face_slice  = slice(0, face_len)
#     pose_slice  = slice(face_len, face_len + pose_len)
#     lhand_slice = slice(face_len + pose_len, face_len + pose_len + hand_len)
#     rhand_slice = slice(face_len + pose_len + hand_len, face_len + pose_len + 2 * hand_len)

#     cap = cv2.VideoCapture(str(in_path))
#     if not cap.isOpened():
#         print(f'Failed to open: {name}')
#         cap.release()
#         continue

#     src_fps     = cap.get(cv2.CAP_PROP_FPS)
#     frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None
#     width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

#     frame_interval = 1
#     out_fps = src_fps if src_fps and src_fps > 0 else 30
#     if TARGET_FPS and src_fps and src_fps > 0:
#         frame_interval = max(int(round(src_fps / TARGET_FPS)), 1)
#         out_fps = float(src_fps) / frame_interval

#     fourcc = cv2.VideoWriter_fourcc(*'mp4v')
#     writer = cv2.VideoWriter(str(out_path), fourcc, out_fps, (width, height))
#     if not writer.isOpened():
#         out_path = OUT_DIR / f'{name}_overlay.avi'
#         fourcc = cv2.VideoWriter_fourcc(*'XVID')
#         writer = cv2.VideoWriter(str(out_path), fourcc, out_fps, (width, height))

#     if not writer.isOpened():
#         print(f'Failed to open writer for: {name}')
#         cap.release()
#         continue

#     frame_idx = 0
#     written   = 0

#     while True:
#         ret, frame = cap.read()
#         if not ret:
#             break

#         if frame_interval > 1 and (frame_idx % frame_interval) != 0:
#             frame_idx += 1
#             continue

#         if frame_count and frame_count > 1:
#             kp_idx = int(round(frame_idx * (T - 1) / max(frame_count - 1, 1)))
#         else:
#             kp_idx = min(frame_idx, T - 1)

#         kp_frame   = keypoints[kp_idx]
#         mask_frame = mask[kp_idx] if mask is not None else None

#         # Face (yellow dots)
#         draw_points(frame, kp_frame[face_slice], None if mask_frame is None else mask_frame[face_slice], (0, 255, 255), radius=1)

#         # Pose (green dots + connections)
#         pose_pts  = kp_frame[pose_slice]
#         pose_mask = None if mask_frame is None else mask_frame[pose_slice]
#         draw_points(frame, pose_pts, pose_mask, (0, 255, 0), radius=2)
#         if POSE_CONNECTIONS is not None and pose_idx is not None:
#             pose_map = {int(idx): i for i, idx in enumerate(pose_idx)}
#             conns = [(pose_map[a], pose_map[b]) for a, b in POSE_CONNECTIONS if a in pose_map and b in pose_map]
#             draw_connections(frame, pose_pts, pose_mask, conns, (0, 255, 0), thickness=1)

#         # Left hand (blue) / Right hand (red)
#         lpts  = kp_frame[lhand_slice]
#         rpts  = kp_frame[rhand_slice]
#         lmask = None if mask_frame is None else mask_frame[lhand_slice]
#         rmask = None if mask_frame is None else mask_frame[rhand_slice]
#         draw_points(frame, lpts, lmask, (255, 0, 0), radius=2)
#         draw_points(frame, rpts, rmask, (0, 0, 255), radius=2)
#         if HAND_CONNECTIONS is not None:
#             draw_connections(frame, lpts, lmask, HAND_CONNECTIONS, (255, 0, 0), thickness=1)
#             draw_connections(frame, rpts, rmask, HAND_CONNECTIONS, (0, 0, 255), thickness=1)

#         writer.write(frame)
#         written   += 1
#         frame_idx += 1

#         if MAX_FRAMES is not None and written >= int(MAX_FRAMES):
#             break

#     cap.release()
#     writer.release()

#     if written == 0:
#         print(f'No frames written for: {name}')
#     else:
#         print(f'Wrote: {out_path.name} ({written} frames)')

# print(f'Done. Overlays saved to: {OUT_DIR.resolve()}')

Overlaying 5 clips
Keypoint dir: C:\My Projects\sign-language-bridge\data\keypoints\train
Output dir: C:\My Projects\sign-language-bridge\data\overlays


Overlay:  20%|██        | 1/5 [00:02<00:09,  2.45s/it]

Wrote: --7E2sU6zP4_10-5-rgb_front_overlay.mp4 (365 frames)


Overlay:  40%|████      | 2/5 [00:04<00:07,  2.39s/it]

Wrote: --7E2sU6zP4_12-5-rgb_front_overlay.mp4 (349 frames)


...

Wrote: --7E2sU6zP4_13-5-rgb_front_overlay.mp4 (144 frames)


Overlay:  80%|████████  | 4/5 [00:07<00:01,  1.61s/it]

Wrote: --7E2sU6zP4_5-5-rgb_front_overlay.mp4 (212 frames)


Overlay: 100%|██████████| 5/5 [00:09<00:00,  1.84s/it]

Wrote: --7E2sU6zP4_7-5-rgb_front_overlay.mp4 (289 frames)
Done. Overlays saved to: C:\My Projects\sign-language-bridge\data\overlays
